## 2. Create Indexer and Index Documents

In this notebook we will index the documents in the blob storage. We won't be using a blob Storage knowledge source since the platform will automate how the index is created, and we want to control that. So we will create the indexer.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.search.documents.indexes import SearchIndexerClient
from azure.search.documents.indexes.models import(
    SearchIndexerDataSourceConnection,
    SearchIndexerDataSourceType,
    SearchIndexerDataContainer,
    SearchIndexerSkillset,
    SplitSkill,
    AzureOpenAIEmbeddingSkill,
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    IndexProjectionMode,
    InputFieldMappingEntry,
    OutputFieldMappingEntry,
    SearchIndexer,
    FieldMapping,
    IndexingParameters
)


load_dotenv(override=True)

# Service endpoints
SEARCH_ENDPOINT = os.environ["SEARCH_ENDPOINT"]
AOAI_ENDPOINT = os.environ["AOAI_ENDPOINT"]

# Model deployments
EMBEDDING_MODEL = os.environ.get("AOAI_EMBEDDING_MODEL", "text-embedding-3-large")
EMBEDDING_DEPLOYMENT = os.environ.get("AOAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
GPT_MODEL = os.environ.get("AOAI_AGENTIC_GPT_MODEL", "gpt-4o-mini")
GPT_DEPLOYMENT = os.environ.get("AOAI_AGENTIC_GPT_DEPLOYMENT", "gpt-4o-mini")

# Resource names
INDEX_MANUALS = "product-manuals"
INDEX_CATALOG = "product-catalog"
INDEXED_SOURCE_MANUALS = "product-index-source-manuals"
INDEXED_SOURCE_CATALOG = "product-index-source-catalog"
KNOWLEDGE_BASE_NAME = "product-kb"

# Storage containers and resource-id connection strings.
# Using ResourceId=... means the indexer authenticates with the Search service managed identity.
product_manuals_container = "product-manuals"
product_catalog_container = "product-catalog"
storage_resource_id_manuals = os.environ.get("STORAGE_RESOURCE_ID_MANUALS", os.environ.get("STORAGE_RESOURCE_ID"))
storage_resource_id_catalog = os.environ.get("STORAGE_RESOURCE_ID_CATALOG", os.environ.get("STORAGE_RESOURCE_ID"))

if not storage_resource_id_manuals:
    raise ValueError("Missing STORAGE_RESOURCE_ID_MANUALS or STORAGE_RESOURCE_ID")
if not storage_resource_id_catalog:
    raise ValueError("Missing STORAGE_RESOURCE_ID_CATALOG or STORAGE_RESOURCE_ID")

credential = DefaultAzureCredential()

print("Configuration loaded for indexing workflow.")

Models: embedding=text-embedding-3-large, chat=gpt-4.1


### 2.1 Create Data Source

We need two data sources, one for each index

In [31]:
indexer_client = SearchIndexerClient(SEARCH_ENDPOINT, credential)

manuals_data_source_connection = SearchIndexerDataSourceConnection(
    name=f"{INDEX_MANUALS}-datasource",
    type=SearchIndexerDataSourceType.AZURE_BLOB,
    connection_string=storage_resource_id_manuals,
    container=SearchIndexerDataContainer(name=product_manuals_container),
)

catalog_data_source_connection = SearchIndexerDataSourceConnection(
    name=f"{INDEX_CATALOG}-datasource",
    type=SearchIndexerDataSourceType.AZURE_BLOB,
    connection_string=storage_resource_id_catalog,
    container=SearchIndexerDataContainer(name=product_catalog_container),
)

indexer_client.create_or_update_data_source_connection(manuals_data_source_connection)
indexer_client.create_or_update_data_source_connection(catalog_data_source_connection)

print(f"✅ Data source '{INDEX_MANUALS}-datasource' ready")
print(f"✅ Data source '{INDEX_CATALOG}-datasource' ready")

✅ Data source 'product-manuals-datasource' ready
✅ Data source 'product-catalog-datasource' ready


### 2.2 Create Skills

For the product manuals:

In [41]:
SKILLSET_MANUALS = f"{INDEX_MANUALS}-skillset"

# Projection-based skillset aligned to manuals index schema:
# id (key), parent_id (parent key), title, content, blob_url, content_embedding
# page/page_number_int are backfilled after indexing from chunk key suffix (_pages_<n>).
manuals_skillset = SearchIndexerSkillset(
    name=SKILLSET_MANUALS,
    description=f"Skillset for index '{INDEX_MANUALS}'",
    skills=[
        SplitSkill(
            text_split_mode="pages",
            context="/document",
            maximum_page_length=2000,
            page_overlap_length=500,
            inputs=[
                InputFieldMappingEntry(name="text", source="/document/content")
            ],
            outputs=[
                OutputFieldMappingEntry(name="textItems", target_name="pages"),
                OutputFieldMappingEntry(name="ordinalPositions", target_name="page_numbers"),
            ],
        ),
        AzureOpenAIEmbeddingSkill(
            context="/document/pages/*",
            resource_url=AOAI_ENDPOINT,
            api_key=None,
            deployment_name=EMBEDDING_DEPLOYMENT,
            model_name=EMBEDDING_MODEL,
            dimensions=3072,
            inputs=[
                InputFieldMappingEntry(name="text", source="/document/pages/*")
            ],
            outputs=[
                OutputFieldMappingEntry(
                    name="embedding", target_name="content_embedding"
                )
            ],
        ),
    ],
    index_projection=SearchIndexerIndexProjection(
        selectors=[
            SearchIndexerIndexProjectionSelector(
                target_index_name=INDEX_MANUALS,
                parent_key_field_name="parent_id",
                source_context="/document/pages/*",
                mappings=[
                    InputFieldMappingEntry(
                        name="content",
                        source="/document/pages/*",
                    ),
                    InputFieldMappingEntry(
                        name="content_embedding",
                        source="/document/pages/*/content_embedding",
                    ),
                    InputFieldMappingEntry(
                        name="title",
                        source="/document/metadata_storage_name",
                    ),
                    InputFieldMappingEntry(
                        name="blob_url",
                        source="/document/metadata_storage_path",
                    ),
                ],
            )
        ],
        parameters=SearchIndexerIndexProjectionsParameters(
            projection_mode=IndexProjectionMode.SKIP_INDEXING_PARENT_DOCUMENTS
        ),
    ),
)

indexer_client = SearchIndexerClient(SEARCH_ENDPOINT, credential)
indexer_client.create_or_update_skillset(manuals_skillset)
print(f"✅ Skillset '{SKILLSET_MANUALS}' ready")

✅ Skillset 'product-manuals-skillset' ready


For the product catalog:

In [16]:
SKILLSET_CATALOG = f"{INDEX_CATALOG}-skillset"

# Catalog JSON docs are already structured and small, so no chunking is needed.
# This skill generates ProductVector from ProductDescription.
catalog_skillset = SearchIndexerSkillset(
    name=SKILLSET_CATALOG,
    description=f"Skillset for index '{INDEX_CATALOG}'",
    skills=[
        AzureOpenAIEmbeddingSkill(
            context="/document",
            resource_url=AOAI_ENDPOINT,
            api_key=None,
            deployment_name=EMBEDDING_DEPLOYMENT,
            model_name=EMBEDDING_MODEL,
            dimensions=3072,
            inputs=[
                InputFieldMappingEntry(
                    name="text",
                    source="/document/ProductDescription",
                )
            ],
            outputs=[
                OutputFieldMappingEntry(
                    name="embedding",
                    target_name="ProductVector",
                )
            ],
        ),
    ],
)

indexer_client = SearchIndexerClient(SEARCH_ENDPOINT, credential)
indexer_client.create_or_update_skillset(catalog_skillset)
print(f"✅ Skillset '{SKILLSET_CATALOG}' ready")
print("Note: Ensure your catalog indexer maps /document/ProductVector to index field ProductVector.")

✅ Skillset 'product-catalog-skillset' ready
Note: Ensure your catalog indexer maps /document/ProductVector to index field ProductVector.


### 2.3 Create Indexers

Indexer por product manuals

In [50]:
MANUALS_INDEXER = f"{INDEX_MANUALS}-indexer"

# Manuals indexer: datasource -> skillset (split + embeddings + index projection) -> manuals index.
manuals_indexer = SearchIndexer(
    name=MANUALS_INDEXER,
    data_source_name=f"{INDEX_MANUALS}-datasource",
    target_index_name=INDEX_MANUALS,
    skillset_name=SKILLSET_MANUALS,
)

indexer_client.create_or_update_indexer(manuals_indexer)
print(f"✅ Indexer '{MANUALS_INDEXER}' ready")

✅ Indexer 'product-manuals-indexer' ready


In [ ]:
CATALOG_INDEXER = f"{INDEX_CATALOG}-indexer"

# Map source JSON fields + enriched vector output to the catalog index.
catalog_indexer = SearchIndexer(
    name=CATALOG_INDEXER,
    data_source_name=f"{INDEX_CATALOG}-datasource",
    target_index_name=INDEX_CATALOG,
    skillset_name=SKILLSET_CATALOG,
    parameters=IndexingParameters(
        configuration={
            # Use jsonArray for files containing an array of product objects.
            "parsingMode": "jsonArray",
        }
    ),
    field_mappings=[
        FieldMapping(source_field_name="ProductID", target_field_name="ProductID"),
        FieldMapping(source_field_name="ProductName", target_field_name="ProductName"),
        FieldMapping(source_field_name="ProductCategory", target_field_name="ProductCategory"),
        FieldMapping(source_field_name="Price", target_field_name="Price"),
        FieldMapping(source_field_name="ProductDescription", target_field_name="ProductDescription"),
        FieldMapping(source_field_name="ProductPunchLine", target_field_name="ProductPunchLine"),
        FieldMapping(source_field_name="ImageURL", target_field_name="ImageURL"),
        # Blob indexer metadata path for the source blob.
        FieldMapping(source_field_name="metadata_storage_path", target_field_name="blob_url"),
    ],
    output_field_mappings=[
        FieldMapping(source_field_name="/document/ProductVector", target_field_name="ProductVector"),
    ],
)

indexer_client.create_or_update_indexer(catalog_indexer)
print(f"✅ Indexer '{CATALOG_INDEXER}' ready")
print("Mapped fields: ProductID, ProductName, ProductCategory, Price, ProductDescription, ProductPunchLine, ImageURL, blob_url, ProductVector")
print("If each blob is a single JSON object (not an array), change parsing mode to 'json'.")

Removed existing indexer 'product-catalog-indexer'
✅ Indexer 'product-catalog-indexer' ready
Mapped fields: ProductID, ProductName, ProductCategory, Price, ProductDescription, ProductPunchLine, ImageURL, blob_url, ProductVector
If each blob is a single JSON object (not an array), change parsing mode to 'json'.


### 2.4 Normalize Page Value Per Chunk

Run this **after each manuals indexer run**.

It updates `page` to a single value per chunk using the chunk key suffix (`_pages_<n>`).
This cell uses **merge** updates so existing fields (including vectors) are preserved.

In [54]:
import re
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

PAGE_FIELD = "page"
PAGE_OFFSET = 0  # Use 1 if you want 1-based page numbering.

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
manuals_index = index_client.get_index(INDEX_MANUALS)
field_names = {field.name for field in manuals_index.fields}

if PAGE_FIELD not in field_names:
    raise ValueError(f"Missing '{PAGE_FIELD}' in index '{INDEX_MANUALS}'.")

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_MANUALS,
    credential=credential,
)

page_pattern = re.compile(r"_pages_(\d+)$")

select_fields = ["id", PAGE_FIELD]

batch_size = 1000
skip = 0
updates = []

while True:
    docs = list(
        search_client.search(
            search_text="*",
            select=select_fields,
            top=batch_size,
            skip=skip,
        )
    )
    if not docs:
        break

    for doc in docs:
        match = page_pattern.search(doc.get("id", ""))
        if not match:
            continue

        page_value = int(match.group(1)) + PAGE_OFFSET
        page_str = str(page_value)

        same_page = str(doc.get(PAGE_FIELD)) == page_str
        if same_page:
            continue

        # Use merge to avoid overwriting fields not included in payload (for example vectors).
        payload = {
            "id": doc["id"],
            PAGE_FIELD: page_str,
        }

        updates.append(payload)

    skip += len(docs)
    if len(docs) < batch_size:
        break

if not updates:
    print("No page updates needed.")
else:
    succeeded = 0
    failed = 0
    for i in range(0, len(updates), batch_size):
        chunk = updates[i:i + batch_size]
        result = search_client.merge_documents(documents=chunk)
        succeeded += sum(1 for r in result if r.succeeded)
        failed += sum(1 for r in result if not r.succeeded)

    print(f"Prepared updates: {len(updates)}")
    print(f"Succeeded: {succeeded}, Failed: {failed}")
    print("Done. 'page' is now a single value per chunk, vectors preserved.")

Prepared updates: 162
Succeeded: 162, Failed: 0
Done. 'page' is now a single value per chunk, vectors preserved.
